# Raster analysis with xarray, rasterio, and scipy

This notebook demonstrates how to work with raster data in Python using
**xarray**, **rasterio**, **scipy**, and **matplotlib** (with the interactive
**ipympl** backend).  We build a synthetic elevation model, perform basic
terrain analysis, and overlay the tree-species observation points from the
companion dataset.

## Contents
1. [Create a synthetic DEM with numpy and xarray](#1-create-a-synthetic-dem-with-numpy-and-xarray)
2. [Write and read the raster with rasterio](#2-write-and-read-the-raster-with-rasterio)
3. [Terrain statistics with scipy](#3-terrain-statistics-with-scipy)
4. [Interactive visualisation with ipympl](#4-interactive-visualisation-with-ipympl)
5. [Overlay species observations on the DEM](#5-overlay-species-observations-on-the-dem)

In [ ]:
%matplotlib widget

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import geopandas as gpd
from shapely.geometry import Point
from scipy import ndimage, stats

# Reproducibility
rng = np.random.default_rng(42)

## 1. Create a synthetic DEM with numpy and xarray

We generate a 200 × 200 grid covering Central Europe (roughly 7–16 °E, 47–55 °N)
using superimposed sinusoids and random noise to mimic realistic terrain.

In [ ]:
# Grid dimensions and bounding box
nrows, ncols = 200, 200
lon_min, lon_max = 7.0, 16.0
lat_min, lat_max = 47.0, 55.0

lons = np.linspace(lon_min, lon_max, ncols)
lats = np.linspace(lat_max, lat_min, nrows)  # north-to-south (top row = max lat)

lon_grid, lat_grid = np.meshgrid(lons, lats)

# Base elevation: low in the north, higher towards the Alps in the south
base = 800 * ((lat_max - lat_grid) / (lat_max - lat_min)) ** 2

# Add terrain texture with sinusoids
texture = (
    300 * np.sin(3 * np.pi * (lon_grid - lon_min) / (lon_max - lon_min))
    * np.cos(2 * np.pi * (lat_max - lat_grid) / (lat_max - lat_min))
    + 150 * np.sin(7 * np.pi * (lon_grid - lon_min) / (lon_max - lon_min))
)

# Smooth random noise
noise_raw = rng.normal(0, 80, (nrows, ncols))
noise = ndimage.gaussian_filter(noise_raw, sigma=4)

elevation = np.clip(base + texture + noise, 0, 2000).astype(np.float32)

# Wrap in an xarray DataArray with named coordinates
dem = xr.DataArray(
    elevation,
    dims=["latitude", "longitude"],
    coords={"latitude": lats, "longitude": lons},
    name="elevation_m",
    attrs={"units": "metres", "long_name": "Synthetic elevation model", "crs": "EPSG:4326"},
)

print(dem)
print(f"\nElevation range: {float(dem.min()):.1f} – {float(dem.max()):.1f} m")

## 2. Write and read the raster with rasterio

We save the DataArray as a GeoTIFF to a temporary file, then read it back — a
typical read/write round-trip you would use with real data.

In [ ]:
# Save to GeoTIFF
tmp_dir = Path(tempfile.mkdtemp())
tif_path = tmp_dir / "synthetic_dem.tif"

transform = from_bounds(lon_min, lat_min, lon_max, lat_max, ncols, nrows)

with rasterio.open(
    tif_path,
    "w",
    driver="GTiff",
    height=nrows,
    width=ncols,
    count=1,
    dtype=elevation.dtype,
    crs=CRS.from_epsg(4326),
    transform=transform,
) as dst:
    dst.write(elevation, 1)

print(f"GeoTIFF written to: {tif_path}")

# Read it back and inspect metadata
with rasterio.open(tif_path) as src:
    dem_array = src.read(1)
    profile = src.profile

print("\nRasterio profile:")
for k, v in profile.items():
    print(f"  {k}: {v}")

## 3. Terrain statistics with scipy

We use `scipy.ndimage` to compute a slope proxy (Sobel gradient magnitude) and
`scipy.stats` to describe the elevation distribution.

In [ ]:
# Slope proxy: Sobel edge magnitude
sx = ndimage.sobel(dem_array, axis=1)
sy = ndimage.sobel(dem_array, axis=0)
slope = np.hypot(sx, sy)

slope_da = xr.DataArray(
    slope,
    dims=["latitude", "longitude"],
    coords={"latitude": lats, "longitude": lons},
    name="slope_proxy",
    attrs={"long_name": "Sobel slope proxy"},
)

# Descriptive statistics
result = stats.describe(dem_array.ravel())
print("Elevation statistics (scipy.stats.describe):")
print(f"  n          = {result.nobs}")
print(f"  min/max    = {result.minmax[0]:.1f} / {result.minmax[1]:.1f} m")
print(f"  mean       = {result.mean:.1f} m")
print(f"  variance   = {result.variance:.1f} m²")
print(f"  skewness   = {result.skewness:.3f}")
print(f"  kurtosis   = {result.kurtosis:.3f}")

# Elevation histogram via xarray + numpy
counts, bin_edges = np.histogram(dem.values.ravel(), bins=30)
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

## 4. Interactive visualisation with ipympl

The `%matplotlib widget` magic at the top of the notebook activates the **ipympl**
backend, so the figures below are interactive — you can zoom and pan them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# — DEM —
dem_plot = axes[0].imshow(
    dem.values,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin="upper",
    cmap="terrain",
    aspect="auto",
)
fig.colorbar(dem_plot, ax=axes[0], label="Elevation (m)")
axes[0].set_title("Synthetic DEM")
axes[0].set_xlabel("Longitude (°E)")
axes[0].set_ylabel("Latitude (°N)")

# — Slope proxy —
slope_plot = axes[1].imshow(
    slope_da.values,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin="upper",
    cmap="hot_r",
    aspect="auto",
)
fig.colorbar(slope_plot, ax=axes[1], label="Slope proxy")
axes[1].set_title("Sobel slope proxy")
axes[1].set_xlabel("Longitude (°E)")

# — Elevation histogram —
axes[2].bar(bin_centres, counts, width=np.diff(bin_edges), color="steelblue", edgecolor="white")
axes[2].set_xlabel("Elevation (m)")
axes[2].set_ylabel("Pixel count")
axes[2].set_title("Elevation histogram")

fig.suptitle("Synthetic DEM analysis", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Overlay species observations on the DEM

We load the species-observation CSV, convert it to a GeoDataFrame, and plot
the points on top of the DEM to see how the sampling relates to the terrain.

In [ ]:
df = pd.read_csv("../data/species_observations.csv")

geometry = [Point(lon, lat) for lon, lat in zip(df["longitude"], df["latitude"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Sample DEM elevation at each observation point
def sample_dem(row):
    lon_idx = int(np.argmin(np.abs(lons - row["longitude"])))
    lat_idx = int(np.argmin(np.abs(lats - row["latitude"])))
    return float(dem.values[lat_idx, lon_idx])

gdf["dem_elevation_m"] = gdf.apply(sample_dem, axis=1)

print("Observed vs DEM elevation (first 5 rows):")
print(gdf[["species", "elevation_m", "dem_elevation_m"]].head())

In [ ]:
species_list = gdf["species"].unique()
cmap_pts = plt.get_cmap("tab10")
colour_map = {sp: cmap_pts(i) for i, sp in enumerate(species_list)}

fig, ax = plt.subplots(figsize=(8, 7))

# DEM background
img = ax.imshow(
    dem.values,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin="upper",
    cmap="terrain",
    aspect="auto",
    alpha=0.85,
)
fig.colorbar(img, ax=ax, label="Elevation (m)", fraction=0.03)

# Observation points
for sp in species_list:
    subset = gdf[gdf["species"] == sp]
    ax.scatter(
        subset["longitude"],
        subset["latitude"],
        s=subset["count"] * 4,
        c=[colour_map[sp]],
        edgecolors="white",
        linewidths=0.6,
        label=sp,
        zorder=3,
    )

ax.set_xlim(lon_min, lon_max)
ax.set_ylim(lat_min, lat_max)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_title("Species observations on synthetic DEM\n(bubble size ∝ observation count)")
ax.legend(fontsize=7, loc="upper left", framealpha=0.8)
plt.tight_layout()
plt.show()

---
**Key packages used in this notebook:**

| Package | Purpose |
|---|---|
| `numpy` | Array creation and numerical operations |
| `xarray` | Labelled multi-dimensional arrays |
| `rasterio` | Reading and writing raster files (GeoTIFF) |
| `scipy` | Terrain filtering (`ndimage`) and statistics (`stats`) |
| `geopandas` | Vector point data with CRS support |
| `pandas` | Tabular data loading and manipulation |
| `matplotlib` + `ipympl` | Interactive notebook figures |

**Next steps:** replace the synthetic DEM with a real SRTM tile, compute
vegetation indices from multispectral imagery, or fit species-distribution
models using `scikit-learn`.